# 4. Digital Signatures — The Flip That Makes Authentication Possible

This is the notebook where the pieces click together. If you've ever felt like you
understand public/private keys individually but not *why* they make login systems
secure, this is usually the missing piece.

By the end of this notebook you should be able to answer:

- What's the difference between "encrypt with a public key" and "sign with a
  private key"? Why are they opposite operations with opposite purposes?
- How does a signature let a stranger verify a message came from you, without
  needing your private key?
- What happens, concretely, if a signed message is tampered with?


## 4.1 The flip

In Notebook 2, we encrypted **with the public key** so that only the **private
key** holder could read the message. That protects **confidentiality** — keeping
content secret from onlookers.

Digital signatures use the *same key pair*, but in the **opposite direction**, for
a completely different purpose:

| Operation | Who can do it | What it proves |
|---|---|---|
| Encrypt with public key → decrypt with private key | Anyone can encrypt; only the private key holder can decrypt | **Confidentiality** — nobody else can read it |
| Sign with private key → verify with public key | Only the private key holder can produce a valid signature; anyone can verify it | **Authenticity & integrity** — this really came from the key holder, and wasn't altered |

This is the flip. Authentication systems (SSO, Okta, JWTs, TLS certificates — all
of it) care almost entirely about the **second** row, not the first. Nobody is
trying to keep a login token secret from you — you can read your own JWT payload
just by looking at it (we'll see this directly in Notebook 5). What matters is that
a **relying party** (an app you're logging into) can be sure the token really was
issued by the identity provider it trusts, and that nobody tampered with it in
transit.


## 4.2 Generating a signing key pair


In [1]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes

private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()


## 4.3 Signing a message

To sign, the private key holder first hashes the message (this is where Notebook 3
comes back), then encrypts *that hash* using the private key. The result is the
signature. In practice, the `cryptography` library does the hash-then-sign step
for you in one call.


In [2]:
message = b"Identity Provider confirms: user alice@example.com successfully authenticated at 09:41 UTC."

signature = private_key.sign(
    message,
    padding.PKCS1v15(),
    hashes.SHA256(),
)

print("Signature (base64-ish bytes, not human-readable):")
print(signature[:64], "...")


Signature (base64-ish bytes, not human-readable):
b'\x00=\x8f\xfe*)\xdeG\x8c\xb2\xe5\xd7SS\xcc\x11\r\xbb\xff\x11:K_\x9eK\x12\xff\x02\xd7j\x12#\xf8/\x91\xc9\xac1U\x0eb$\xd8N]}m6uIM\xdb\x04yA\xbb7\xd4\xf2\xae\x0fr\xd0+' ...


## 4.4 Verifying the signature — with only the public key

Now imagine a completely different party — a relying party who has never talked to
the private key holder directly, but *has* obtained their public key (for example,
by fetching it from a well-known URL, which is exactly what real identity
providers publish). They can verify the message is authentic and untampered,
**without ever touching the private key**.


In [3]:
try:
    public_key.verify(
        signature,
        message,
        padding.PKCS1v15(),
        hashes.SHA256(),
    )
    print("✅ Signature is VALID — this message really was signed by the private key holder, unaltered.")
except Exception as e:
    print("❌ Signature is INVALID:", e)


✅ Signature is VALID — this message really was signed by the private key holder, unaltered.


## 4.5 Tamper with the message and watch verification fail

This is the moment that matters most. Let's change a single character in the
message — as if an attacker intercepted it and modified the claimed username — and
try to verify the **original signature** against the **modified message**.


In [4]:
tampered_message = b"Identity Provider confirms: user MALLORY@example.com successfully authenticated at 09:41 UTC."

try:
    public_key.verify(
        signature,
        tampered_message,
        padding.PKCS1v15(),
        hashes.SHA256(),
    )
    print("Signature verified (this should NOT happen!)")
except Exception as e:
    print("❌ Signature correctly rejected as INVALID.")
    print("   Reason:", type(e).__name__)
    print("   The signature only matches the exact original message it was created for.")


❌ Signature correctly rejected as INVALID.
   Reason: InvalidSignature
   The signature only matches the exact original message it was created for.


This is the entire security model of every token-based authentication system in
this repository (and in Okta, Auth0, and virtually every OIDC/SAML identity
provider in production use): **a relying party trusts a token not because it
arrived over a "secure-looking" channel, but because its signature can only have
been produced by whoever holds the matching private key — and any tampering breaks
the signature.**


## 4.6 What this means operationally

- The identity provider (Okta, Auth0, or the toy IdP we'll build in Part 2) keeps
  a private key that **never leaves its servers**.
- The identity provider publishes the matching **public key** at a well-known
  location, so any relying party can fetch it and verify tokens independently —
  without ever calling the identity provider back to ask "is this real?" (that
  callback-free property is one reason token-based auth scales so well).
- If that private key is ever leaked, an attacker could forge tokens claiming to
  be anyone — which is why an identity provider's private key is one of the
  highest-value secrets in an entire organization's infrastructure.

This also directly answers your likely original question: **the "security" of the
whole login flow rests on exactly one thing** — that the private key stays
private. Everything else (public keys, published endpoints, even the tokens
themselves) is, by design, allowed to be public.


## Summary

- Signing (private key) and verifying (public key) is the *opposite* operation
  from encrypting (public key) and decrypting (private key) — and it serves a
  different purpose: authenticity and integrity, not secrecy.
- A signature is produced from a hash of the message, then encrypted with the
  private key.
- Anyone holding the public key can verify a signature — but only the private key
  holder could have produced a valid one.
- Tampering with a signed message, even slightly, causes verification to fail.
- This one mechanism is the security foundation of JWTs, OIDC, SAML, and
  effectively every SSO system you'll ever use.

**Next:** `05_building_a_jwt_by_hand.ipynb`
